# Amazon Berkeley Metadata Query

This notebook is used viewing and creating the baseline ablation dataset using the Amazon Berkeley Dataset.

This includes downloading metadata, selecting a subset, downloading 360 degree images, downloading 3D parts, and converting them to meshes.

Documents on how to use the dataset can be found in docs.
Utilities for easily calling the dataset can be found in utils.

In [ ]:
#imports
import pandas as pd
import requests
import trimesh
from PIL import Image

import tarfile
import gzip
import json
from pathlib import Path
import time

from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://amazon-berkeley-objects.s3.amazonaws.com"
DATA_DIR = Path("data/abo")
METADATA_DIR = DATA_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "abo-listings.tar": "archives/abo-listings.tar",
    "spins.csv.gz": "spins/metadata/spins.csv.gz",
    "3dmodels.csv.gz": "3dmodels/metadata/3dmodels.csv.gz"
}

In [2]:
#gather metadata
for filename, s3_path in FILES.items():
    target = METADATA_DIR / filename

    if target.exists():
        print(f"Already downloaded: {filename}")
        continue

    url = f"{BASE_URL}/{s3_path}"
    print(f"Downloading {filename}...")

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()

        with open(target, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    print(f"Saved: {target}")

Already downloaded: abo-listings.tar
Already downloaded: spins.csv.gz
Already downloaded: 3dmodels.csv.gz


In [3]:
#load product listings and metadata
def first_value(data):
    if isinstance(data, str):
        return data
    if isinstance(data, list):
        for entry in data:
            if isinstance(entry, dict) and entry.get("value") is not None:
                return str(entry["value"])
    return ""


records = []

with tarfile.open(METADATA_DIR / "abo-listings.tar", "r") as archive:
    for member in archive:
        if not member.isfile() or "listings/metadata/" not in member.name:
            continue
        if not member.name.endswith(".json.gz"):
            continue

        with archive.extractfile(member) as src:
            with gzip.GzipFile(fileobj=src) as f:
                for line in f:
                    item = json.loads(line)
                    records.append({
                        "item_id": item.get("item_id", ""),
                        "product_type": first_value(item.get("product_type")),
                        "title": first_value(item.get("item_name")),
                        "spin_id": item.get("spin_id") or "",
                        "3dmodel_id": item.get("3dmodel_id") or ""
                    })

listings = pd.DataFrame(records)
spins = pd.read_csv(METADATA_DIR / "spins.csv.gz")
models = pd.read_csv(METADATA_DIR / "3dmodels.csv.gz")

print("Listings:", len(listings))
print("Spin images:", len(spins))
print("3D model records:", len(models))

display(listings.head())

Listings: 147702
Spin images: 586584
3D model records: 7953


,item_id,product_type,title,spin_id,3dmodel_id
0,B07NQ437BB,SHOES,"Amazon Essentials Manny Footwear, Peau, 11 M US",,
1,B0857LSVB7,CELLULAR_PHONE_CASE,Amazon Brand - Solimo Designer Lion UV Printed...,,
2,B07C5FF8QS,AUTO_OIL,"AmazonBasics High Mileage Motor Oil, Synthetic...",,
3,B07K591232,HOME_BED_AND_BATH,AmazonBasics piumino alternativo,,
4,B07TG425LX,CELLULAR_PHONE_CASE,Amazon Brand - Solimo Designer Semi Circle Tex...,,


In [4]:
# Verify complete 72-image sequences
spin_stats = spins.groupby("spin_id")["azimuth"].agg(
    ["size", "nunique", "min", "max"]
)

valid_spins = spin_stats[
    (spin_stats["size"] == 72) &
    (spin_stats["nunique"] == 72) &
    (spin_stats["min"] == 0) &
    (spin_stats["max"] == 71)
].index

listings["has_72_images"] = listings["spin_id"].isin(valid_spins)
listings["has_gt_model"] = listings["3dmodel_id"].isin(models["3dmodel_id"])

eligible = listings[
    listings["has_72_images"] &
    listings["has_gt_model"]
].drop_duplicates("spin_id").copy()

print("Products with 72 images:", listings["has_72_images"].sum())
print("Products with GT models:", listings["has_gt_model"].sum())
print("Products with BOTH:", len(eligible))

Products with 72 images: 8127
Products with GT models: 7960
Products with BOTH: 7671


In [5]:
#check catagories
category_counts = eligible["product_type"].value_counts()

display(category_counts.head(30).to_frame("product_count"))

,product_count
product_type,
CHAIR,1155
SOFA,809
RUG,803
WALL_ART,595
TABLE,547
HOME_FURNITURE_AND_DECOR,508
LAMP,341
STOOL_SEATING,336
OTTOMAN,285


In [6]:
#choose representative catagories
categories = ["CHAIR", "DRESSER", "SOFA", "LAMP", "VASE"]

display(category_counts.reindex(categories, fill_value=0))

product_type
CHAIR      1155
DRESSER      58
SOFA        809
LAMP        341
VASE         50
Name: count, dtype: int64

In [7]:
selected = pd.concat([
    eligible[eligible["product_type"] == category].sample(
        n=20,
        random_state=42
    )
    for category in categories
])

selected.to_csv(DATA_DIR / "selected_products.csv", index=False)

In [8]:
#create .csv of image manifests 
# Get all images for our selected products
frames = spins[
    spins["spin_id"].isin(selected["spin_id"])
].copy()

frames = frames.sort_values(["spin_id", "azimuth"])

# Assign indices 0-71 to each product
frames["image_index"] = frames.groupby("spin_id").cumcount()

# Assign ablation subsets
frames["use_72"] = True
frames["use_36"] = frames["image_index"] % 2 == 0
frames["use_18"] = frames["image_index"] % 4 == 0
frames["use_9"] = frames["image_index"] % 8 == 0

# Add product information
frames = frames.merge(
    selected[["item_id", "product_type", "spin_id"]],
    on="spin_id",
    how="left"
)

# Save manifest
frames.to_csv(DATA_DIR / "image_manifest.csv", index=False)

print(f"Products: {frames['spin_id'].nunique()}")
print(f"Total images to download: {len(frames)}")

display(frames.head())

Products: 100
Total images to download: 7200


,spin_id,azimuth,image_id,height,width,path,image_index,use_72,use_36,use_18,use_9,item_id,product_type
0,07ae9e54,0,71RI2cRp+cL,780,1179,07/07ae9e54/07ae9e54_00.jpg,0,True,True,True,True,B07HSNQW37,DRESSER
1,07ae9e54,1,71viInbhQfL,780,1179,07/07ae9e54/07ae9e54_01.jpg,1,True,False,False,False,B07HSNQW37,DRESSER
2,07ae9e54,2,71q9ri1ph8L,780,1179,07/07ae9e54/07ae9e54_02.jpg,2,True,True,False,False,B07HSNQW37,DRESSER
3,07ae9e54,3,71Z74ejK1PL,780,1179,07/07ae9e54/07ae9e54_03.jpg,3,True,False,False,False,B07HSNQW37,DRESSER
4,07ae9e54,4,71hqUSPSqhL,780,1179,07/07ae9e54/07ae9e54_04.jpg,4,True,True,True,False,B07HSNQW37,DRESSER


In [9]:
print("Images per experiment:")

for count in [72, 36, 18, 9]:
    subset = frames[frames[f"use_{count}"]]

    print(f"{count}: {len(subset)} images")
    print(f"Indices: {sorted(subset['image_index'].unique())}")

Images per experiment:
72: 7200 images
Indices: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(

In [ ]:
#image download
manifest = pd.read_csv(DATA_DIR / "image_manifest.csv")

BASE_URL = "https://amazon-berkeley-objects.s3.amazonaws.com/spins/original"

def download_image(row):
    destination = (
        DATA_DIR / "images" /
        row.product_type /
        row.spin_id /
        f"{int(row.azimuth):02d}.jpg"
    )

    if destination.exists() and destination.stat().st_size > 0:
        return "skipped", str(destination)

    destination.parent.mkdir(parents=True, exist_ok=True)
    url = f"{BASE_URL}/{row.path}"
    temp = destination.with_suffix(".part")

    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=60) as response:
                response.raise_for_status()

                with open(temp, "wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            temp.replace(destination)
            return "downloaded", str(destination)

        except (requests.RequestException, OSError) as e:
            temp.unlink(missing_ok=True)
            if attempt == 2:
                return "failed", f"{url}: {e}"
            time.sleep(2 * (attempt + 1))


results = {"downloaded": 0, "skipped": 0, "failed": 0}
failures = []

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [
        executor.submit(download_image, row)
        for row in manifest.itertuples(index=False)
    ]

    for i, future in enumerate(as_completed(futures), 1):
        status, detail = future.result()
        results[status] += 1

        if status == "failed":
            failures.append(detail)

        if i % 100 == 0:
            print(f"{i}/{len(futures)} processed")

print(results)

if failures:
    print("Failed downloads:", failures[:10])

100/7200 processed
200/7200 processed
300/7200 processed
400/7200 processed
500/7200 processed
600/7200 processed
700/7200 processed
800/7200 processed
900/7200 processed
1000/7200 processed
1100/7200 processed
1200/7200 processed
1300/7200 processed
1400/7200 processed
1500/7200 processed
1600/7200 processed
1700/7200 processed
1800/7200 processed
1900/7200 processed
2000/7200 processed
2100/7200 processed
2200/7200 processed
2300/7200 processed
2400/7200 processed
2500/7200 processed
2600/7200 processed
2700/7200 processed
2800/7200 processed
2900/7200 processed
3000/7200 processed
3100/7200 processed
3200/7200 processed
3300/7200 processed
3400/7200 processed
3500/7200 processed
3600/7200 processed
3700/7200 processed
3800/7200 processed
3900/7200 processed
4000/7200 processed
4100/7200 processed
4200/7200 processed
4300/7200 processed
4400/7200 processed
4500/7200 processed
4600/7200 processed
4700/7200 processed
4800/7200 processed
4900/7200 processed
5000/7200 processed
5100/7200

In [11]:
#validate image download
missing = []

for row in manifest.itertuples(index=False):
    path = (
        DATA_DIR / "images" /
        row.product_type /
        row.spin_id /
        f"{int(row.azimuth):02d}.jpg"
    )

    if not path.is_file() or path.stat().st_size == 0:
        missing.append(str(path))

print(f"Expected images: {len(manifest)}")
print(f"Downloaded images: {len(manifest) - len(missing)}")
print(f"Missing images: {len(missing)}")

Expected images: 7200
Downloaded images: 7200
Missing images: 0


In [ ]:
#3D model download 
MODEL_URL = "https://amazon-berkeley-objects.s3.amazonaws.com/3dmodels/original"
MODEL_DIR = DATA_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Match selected products to model metadata
selected_models = selected[["3dmodel_id"]].drop_duplicates().merge(
    models[["3dmodel_id", "path"]],
    on="3dmodel_id",
    how="left"
)

assert selected_models["path"].notna().all(), "Some models have no matching path"

def download_model(row):
    destination = MODEL_DIR / f"{row['3dmodel_id']}.glb"

    if destination.exists() and destination.stat().st_size > 0:
        return "skipped"

    url = f"{MODEL_URL}/{row['path']}"
    temp = destination.with_suffix(".part")

    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=120) as response:
                response.raise_for_status()

                with open(temp, "wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            temp.replace(destination)
            return "downloaded"

        except (requests.RequestException, OSError):
            temp.unlink(missing_ok=True)
            time.sleep(2)

    print(f"Failed: {row['3dmodel_id']}")
    return "failed"


results = {"downloaded": 0, "skipped": 0, "failed": 0}

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(download_model, row)
        for _, row in selected_models.iterrows()
    ]

    for future in as_completed(futures):
        results[future.result()] += 1

print(results)

{'downloaded': 0, 'skipped': 100, 'failed': 0}


In [ ]:
#convert .glb to meshes

MODEL_DIR = DATA_DIR / "models"
MESH_DIR = DATA_DIR / "meshes"
MESH_DIR.mkdir(parents=True, exist_ok=True)

for glb_path in MODEL_DIR.glob("*.glb"):
    # Load GLB scene
    scene = trimesh.load_scene(glb_path)

    # Combine all mesh components, applying scene transforms
    mesh = scene.to_mesh()

    # Save geometry without textures/materials
    mesh.export(MESH_DIR / f"{glb_path.stem}.ply")

print(f"Converted {len(list(MESH_DIR.glob('*.ply')))} meshes")

Converted 100 meshes


In [ ]:
#view sample mesh for validation
mesh = trimesh.load_mesh(next(MESH_DIR.glob("*.ply")))

print("Vertices:", len(mesh.vertices))
print("Faces:", len(mesh.faces))
print("Dimensions:", mesh.extents)
print("Watertight:", mesh.is_watertight)
print("Surface area:", mesh.area)

mesh.show()

Vertices: 11566
Faces: 20790
Dimensions: [1.72228324 0.88837576 0.87345701]
Watertight: False
Surface area: 11.02756779125221
